# Fine-tuning Embedding Models

In the following Notebook we will be exploring one of the most powerful techniques to take your single-domain RAG pipelines to the next level...

**Fine-tuning the Embeddings Model!**

---
In this notebook, we'll worth through the following tasks:

- 🤝 Breakout Room #1:
  1. Dependencies and Boilerplate
  2. Loading Data
  3. Constructing a Fine-tuning Dataset
  4. Fine-tuning `all-MiniLM-L6-v2`
  5. Evaluating our Retriever

## Task 1: Dependencies and Boilerplate

We'll set up our `nest_asyncio` so we can leverage async loops in our Notebook.

We'll also install the required libraries we'll be using today, and set up our OpenAI API key!

### Nest Asyncio

In [1]:
import nest_asyncio

nest_asyncio.apply()

### Install Dependencies

In [2]:
!pip install -qU langchain_openai langchain_huggingface langchain_core langchain langchain_community


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
!pip install -qU pymupdf faiss-cpu


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


### Provide OpenAI API Key

In [4]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
#os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter Your OpenAI API Key: ")

## Task 2: Loading Data

The data can be found in [this GitHub repo](https://github.com/AI-Maker-Space/DataRepository/tree/main/high-performance-rag).

In [5]:
#!git clone https://github.com/AI-Maker-Space/DataRepository.git

In [6]:
#%cd DataRepository

In [7]:
from langchain_community.document_loaders import PyMuPDFLoader
pdf_path = "./data/fed-monetary-policy.pdf"
training_documents_loaded = PyMuPDFLoader(pdf_path)

/Users/katerinag/early-warnings/Model-Fine-tuning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 250,
    chunk_overlap  = 20,
    length_function = len
)

In [9]:
training_documents = text_splitter.split_documents(training_documents_loaded.load())

In [10]:
len(training_documents)

232

In [11]:
import uuid

id_set = set()

for document in training_documents:
  id = str(uuid.uuid4())
  while id in id_set:
    id = uuid.uuid4()
  id_set.add(id)
  document.metadata["id"] = id

In [12]:
training_split_documents = training_documents[:50]

In [13]:
val_split_documents = training_documents[50:100]

In [14]:
test_split_documents = training_documents[100:150]

## Task 3: Constructing a Fine-tuning Dataset

Using the nodes we created above, we can finally start constructing a fine-tuning dataset utilizing OpenAI's `gpt-4o-mini` (released [today](https://openai.com/index/gpt-4o-mini-advancing-cost-efficient-intelligence/)).

The basic idea here is straightforward enough:

1. We look at a document
2. We generate questions that could be answered by that node

This gives us a number of question/context pairs that we can use to fine-tune our Embeddings model.

In [15]:
from langchain_openai import ChatOpenAI

qa_chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

We'll create a simple Question Generation prompt to query `gpt-4o-mini` to generate Questions for each retrieved context.

In [16]:
from langchain_core.prompts import ChatPromptTemplate

qa_prompt = """\
Given the following context, you must generate questions based on only the provided context.

You are to generate {n_questions} questions which should be provided in the following format:

1. QUESTION #1
2. QUESTION #2
...

Context:
{context}
"""

qa_prompt_template = ChatPromptTemplate.from_template(qa_prompt)

We'll create a simple chain to query the LLM!

In [17]:
question_generation_chain = qa_prompt_template | qa_chat_model

There's a lot going on in this function - let's take a deeper look:

1. First, we provide a list of documents and a number of questions
2. We, for each document in our list, generate `n_questions` of questions.
3. We then associate those questions and contexts via a `UUID`.

> NOTE: The reason we're doing this `UUID` association is for ease of use later in the notebook.

In [18]:
import tqdm

def create_questions(documents, n_questions):
  questions = {}
  relevant_docs = {}
  for document in tqdm.tqdm(documents):
    document_content = {"context" : document.page_content, "questions" : []}
    questions_generated = question_generation_chain.invoke({"context": document.page_content, "n_questions": n_questions})
    for question in questions_generated.content.split("\n"):
      question_id = str(uuid.uuid4())
      questions[question_id] = "".join(question.split(".")[1:]).strip()
      relevant_docs[question_id] = [document.metadata["id"]]
  return questions, relevant_docs

We'll use the function to generate training, validation, and test data.

In [19]:
training_questions, training_relevant_contexts = create_questions(training_split_documents, 2)

100%|██████████| 50/50 [01:13<00:00,  1.47s/it]


In [20]:
val_questions, val_relevant_contexts = create_questions(val_split_documents, 2)

100%|██████████| 50/50 [01:30<00:00,  1.82s/it]


In [21]:
test_questions, test_relevant_contexts = create_questions(test_split_documents, 2)

100%|██████████| 50/50 [01:06<00:00,  1.34s/it]


We'll save each dataset for use late - if it's required.

> NOTE: These datasets will be provided in the repository in case you run into any issues with the data generation steps or you wish to save API calls.

In [22]:
import json

training_corpus = {train_item.metadata["id"] : train_item.page_content for train_item in training_split_documents}

train_dataset = {
    "questions" : training_questions,
    "relevant_contexts" : training_relevant_contexts,
    "corpus" : training_corpus
}

with open("training_dataset.jsonl", "w") as f:
  json.dump(train_dataset, f)

In [23]:
val_corpus = {val_item.metadata["id"] : val_item.page_content for val_item in val_split_documents}

val_dataset = {
    "questions" : val_questions,
    "relevant_contexts" : val_relevant_contexts,
    "corpus" : val_corpus
}

with open("val_dataset.jsonl", "w") as f:
  json.dump(val_dataset, f)

In [24]:
train_corpus = {test_item.metadata["id"] : test_item.page_content for test_item in test_split_documents}

test_dataset = {
    "questions" : test_questions,
    "relevant_contexts" : test_relevant_contexts,
    "corpus" : train_corpus
}

with open("test_dataset.jsonl", "w") as f:
  json.dump(test_dataset, f)

## Task 4: Fine-tuning `all-MiniLM-L6-v2`

Now that we have a dataset, let's grab a `sentence-transformers` embeddings model!

We use the **MiniLM-L6-v2** sentence embedding architecture, loaded **only from disk** (no Hugging Face Hub in this notebook).

It already performs well out of the box, but domain-specific terms and vocabulary in our corpus can still benefit from fine-tuning on our synthetic question–context pairs.

Place the base weights under **`models/all-MiniLM-L6-v2/`** (see `STEP0_save_embedding_model_locally.py` if you need to populate that folder once on a networked machine). After training, fine-tuned weights are written to **`models/finetuned_minilm/`**.

In [25]:
!pip install -qU sentence_transformers datasets pyarrow


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


### Local base model (required)

This step assumes **`models/all-MiniLM-L6-v2/`** already contains a saved Sentence Transformers bundle (`modules.json` and the weight files). If it is missing, run **`STEP0_save_embedding_model_locally.py`** once (outside this notebook) on a machine that can reach the Hub, then copy the `models/` folder here. From this point on, nothing in this notebook loads models from Hugging Face.

In [26]:
from pathlib import Path

BASE_MODEL_DIR = Path("models") / "all-MiniLM-L6-v2"
if not (BASE_MODEL_DIR / "modules.json").is_file():
    raise FileNotFoundError(
        f"Missing base model at {BASE_MODEL_DIR.resolve()} (expected modules.json). "
        "Run STEP0_save_embedding_model_locally.py first, then copy models/ here."
    )
print("Base model bundle OK:", BASE_MODEL_DIR.resolve())


Base model bundle OK: /Users/katerinag/early-warnings/Model-Fine-tuning/models/all-MiniLM-L6-v2


In [27]:
from pathlib import Path
from sentence_transformers import SentenceTransformer

MODELS_DIR = Path("models")
BASE_MODEL_DIR = MODELS_DIR / "all-MiniLM-L6-v2"
FINETUNED_DIR = MODELS_DIR / "finetuned_minilm"

if not (BASE_MODEL_DIR / "modules.json").is_file():
    raise FileNotFoundError(
        f"Missing base model at {BASE_MODEL_DIR.resolve()}. Run STEP0_save_embedding_model_locally.py or copy models/all-MiniLM-L6-v2/."
    )

model_id = str(BASE_MODEL_DIR.resolve())
model = SentenceTransformer(model_id, local_files_only=True)
print("Loaded base model from (local):", model_id)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12533.39it/s]


Loaded base model from (local): /Users/katerinag/early-warnings/Model-Fine-tuning/models/all-MiniLM-L6-v2


We'll grab some necessary imports from `sentence_transformers` and `torch`.

In [28]:
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from sentence_transformers import InputExample

We're using a toy batch size here to reflect the limited number of examples we have.

Ideally we'd want to use a much larger batch size. (~64+)

In [29]:
BATCH_SIZE = 20

Let's move our dataset into the expected format for training.

In [30]:
corpus = train_dataset['corpus']
queries = train_dataset['questions']
relevant_docs = train_dataset['relevant_contexts']

examples = []
for query_id, query in queries.items():
    doc_id = relevant_docs[query_id][0]
    text = corpus[doc_id]
    example = InputExample(texts=[query, text])
    examples.append(example)

Now we can create a `torch` `DataLoader`!

In [31]:
loader = DataLoader(
    examples, batch_size=BATCH_SIZE
)

Next up, we'll prepare our loss function!

The loss we're using today is called `MultipleNegativesRankingLoss` - you can find more information [here](https://github.com/UKPLab/sentence-transformers/blob/master/sentence_transformers/losses/MultipleNegativesRankingLoss.py).

In [32]:
from sentence_transformers import losses
loss = losses.MultipleNegativesRankingLoss(model)

/var/folders/nx/kgy4p9z11z576hxy16s57rz40000gn/T/ipykernel_29811/1198462914.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses


Now we can set-up our evaluator.

> NOTE: Due to the formatting of our dataset - this is all we have to do!

In [33]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

corpus = val_dataset['corpus']
queries = val_dataset['questions']
relevant_docs = val_dataset['relevant_contexts']

evaluator = InformationRetrievalEvaluator(queries, corpus, relevant_docs)

/var/folders/nx/kgy4p9z11z576hxy16s57rz40000gn/T/ipykernel_29811/2596565707.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator


We'll train this model for 5 epochs, though you could increase this number if we had a significant amount more data.

In [34]:
EPOCHS = 5

It's training time!

> NOTE: We're manually defining a warm-up period here - this is just to provide a smooth ramp into our training!

In [35]:
from pathlib import Path

if "FINETUNED_DIR" not in globals():
    FINETUNED_DIR = Path("models") / "finetuned_minilm"
FINETUNED_DIR.mkdir(parents=True, exist_ok=True)

warmup_steps = int(len(loader) * EPOCHS * 0.1)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path=str(FINETUNED_DIR),
    show_progress_bar=True,
    evaluator=evaluator,
    evaluation_steps=50,
)

/Users/katerinag/early-warnings/Model-Fine-tuning/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Cosine Accuracy@1,Cosine Accuracy@3,Cosine Accuracy@5,Cosine Accuracy@10,Cosine Precision@1,Cosine Precision@3,Cosine Precision@5,Cosine Precision@10,Cosine Recall@1,Cosine Recall@3,Cosine Recall@5,Cosine Recall@10,Cosine Ndcg@10,Cosine Mrr@10,Cosine Map@100
5,No log,No log,0.900000,0.990000,1.000000,1.000000,0.900000,0.330000,0.200000,0.100000,0.900000,0.990000,1.000000,1.000000,0.959781,0.945833,0.945833
10,No log,No log,0.920000,0.990000,1.000000,1.000000,0.920000,0.330000,0.200000,0.100000,0.920000,0.990000,1.000000,1.000000,0.968472,0.957500,0.957500
15,No log,No log,0.920000,0.990000,1.000000,1.000000,0.920000,0.330000,0.200000,0.100000,0.920000,0.990000,1.000000,1.000000,0.968472,0.957500,0.957500
20,No log,No log,0.920000,0.990000,1.000000,1.000000,0.920000,0.330000,0.200000,0.100000,0.920000,0.990000,1.000000,1.000000,0.968472,0.957500,0.957500
25,No log,No log,0.940000,0.990000,1.000000,1.000000,0.940000,0.330000,0.200000,0.100000,0.940000,0.990000,1.000000,1.000000,0.975853,0.967500,0.967500


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 15.88it/s]


## Task 5: Evaluating our Retriever

Now that we have fine-tuned our retriever - let's see if it's worthwhile!

We'll start with some basic imports.

In [36]:
import pandas as pd

from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_core.documents import Document

Now we'll define a function that will help us evaluate our retrieval process.

> NOTE: We're assuming 1 correct document in a "hit".

In [37]:
def evaluate_openai(
    dataset,
    embed_model,
    top_k=5,
    verbose=False,
):
  corpus = dataset['corpus']
  questions = dataset['questions']
  relevant_docs = dataset['relevant_contexts']
  documents = [Document(page_content=content, metadata={"id": doc_id}) for doc_id, content in corpus.items()]
  vectorstore = FAISS.from_documents(documents, embed_model)

  retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})

  eval_results = []
  for id, question in tqdm.tqdm(questions.items()):
    retrieved_nodes = retriever.invoke(question)
    retrieved_ids = [node.metadata["id"] for node in retrieved_nodes]
    expected_id = relevant_docs[id][0]
    is_hit = expected_id in retrieved_ids
    eval_results.append({"id": id, "question": question, "expected_id": expected_id, "is_hit": is_hit})

  return eval_results

#### 🏗️ Activity #1:

Describe what the function is doing in detail!

All that's left to do is evaluate, we'll evaluate our model against:

1. OpenAI's closed source `text-embedding-3-small`
2. The base non-fine-tuned MiniLM loaded from **`models/all-MiniLM-L6-v2/`** (local files only).

Let's see how it stacks up!

### `text-embedding-3-small`

In [38]:
te3_openai = OpenAIEmbeddings(model="text-embedding-3-small")
te3_results = evaluate_openai(test_dataset, te3_openai)

100%|██████████| 100/100 [00:36<00:00,  2.75it/s]


In [39]:
te3_results_df = pd.DataFrame(te3_results)

In [40]:
te3_hit_rate = te3_results_df["is_hit"].mean()
te3_hit_rate

np.float64(0.97)

### `all-MiniLM-L6-v2` (base)

In [41]:
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings

_base = (Path("models") / "all-MiniLM-L6-v2").resolve()
if not (_base / "modules.json").is_file():
    raise FileNotFoundError(f"Missing base model at {_base}")

huggingface_embeddings = HuggingFaceEmbeddings(
    model_name=str(_base),
    model_kwargs={"local_files_only": True},
)
minilm_base_results = evaluate_openai(test_dataset, huggingface_embeddings)


100%|██████████| 100/100 [00:02<00:00, 48.83it/s]


In [42]:
minilm_base_results_df = pd.DataFrame(minilm_base_results)

In [43]:
minilm_base_hit_rate = minilm_base_results_df["is_hit"].mean()
minilm_base_hit_rate

np.float64(0.96)

### `all-MiniLM-L6-v2` (fine-tuned)

In [44]:
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings

_ft = (Path("models") / "finetuned_minilm").resolve()
if not (_ft / "modules.json").is_file():
    raise FileNotFoundError(
        f"Missing fine-tuned model at {_ft}. Run the training cell (model.fit) first."
    )

finetune_embeddings = HuggingFaceEmbeddings(
    model_name=str(_ft),
    model_kwargs={"local_files_only": True},
)
finetune_results = evaluate_openai(test_dataset, finetune_embeddings)

100%|██████████| 100/100 [00:00<00:00, 165.20it/s]


In [45]:
finetune_results_df = pd.DataFrame(finetune_results)

In [46]:
finetune_hit_rate = finetune_results_df["is_hit"].mean()
finetune_hit_rate

np.float64(0.96)

## Conclusion

As you can see - with only a few hundred data points, we're able to increase our embedding model to outperform even OpenAI's closed source solution!